# Task 2: Contracts, nested data and a star schema

Validate the JSONL contract, quarantine invalid and duplicate bookings, flatten nested records, and create a DuckDB star schema with Parquet outputs.

In [1]:
DATA = "bookings.jsonl"

## Part A — Contract and validation

Pydantic models enforce the nested field rules. Cross-field constraints are checked on the complete booking model.

In [2]:
import json
from collections import Counter
from datetime import date, datetime
from decimal import Decimal
from pathlib import Path
from typing import Literal

import duckdb
import pandas as pd
from pydantic import BaseModel, ConfigDict, Field, ValidationError, model_validator


class ContractModel(BaseModel):
    model_config = ConfigDict(extra="ignore")


class Ticket(ContractModel):
    type: Literal["standard", "vip", "student"]
    qty: int = Field(ge=1, strict=True)
    price: Decimal = Field(ge=0)


class Student(ContractModel):
    id: str = Field(pattern=r"^\d{2}-\d{4}$")
    name: str
    faculty: Literal["MET", "IET", "EMS", "Pharmacy", "Management"]


class Event(ContractModel):
    id: str
    title: str
    club: Literal["Robotics", "Debate", "Music", "Football", "Chess"]
    date: date
    venue: str


class Booking(ContractModel):
    booking_id: str
    booked_at: datetime
    student: Student
    event: Event
    tickets: list[Ticket] = Field(min_length=1)
    total: Decimal = Field(ge=0)
    payment: Literal["card", "cash", "wallet"]
    status: Literal["confirmed", "cancelled"]
    promo_code: str | None = None

    @model_validator(mode="after")
    def check_cross_field_rules(self):
        ticket_total = sum(
            (ticket.qty * ticket.price for ticket in self.tickets), Decimal("0")
        )
        if self.total != ticket_total:
            raise ValueError("total must equal the sum of ticket qty times price")
        if self.booked_at.date() >= self.event.date:
            raise ValueError("booked_at must be before the event date")
        return self


def first_validation_reason(error: ValidationError) -> str:
    detail = error.errors()[0]
    location = ".".join(str(part) for part in detail["loc"]) or "booking"
    return f"{location}: {detail['msg']}"

In [3]:
valid_bookings = []
quarantine = []
quarantine_counts = Counter()
seen_valid_booking_ids = set()
raw_documents = []
lines_read = 0

with open(DATA, "r", encoding="utf-8") as source:
    for line_number, line in enumerate(source, start=1):
        lines_read += 1
        try:
            raw = json.loads(line)
        except json.JSONDecodeError:
            booking_id = None
            reason = "invalid JSON"
        else:
            raw_documents.append((line_number, raw))
            booking_id = raw.get("booking_id") if isinstance(raw, dict) else None
            try:
                booking = Booking.model_validate(raw)
            except ValidationError as error:
                reason = first_validation_reason(error)
            else:
                if booking.booking_id in seen_valid_booking_ids:
                    reason = "duplicate booking_id"
                else:
                    seen_valid_booking_ids.add(booking.booking_id)
                    valid_bookings.append(booking)
                    continue

        quarantine.append({"booking_id": booking_id, "reason": reason})
        quarantine_counts[reason] += 1

with open("quarantine.jsonl", "w", encoding="utf-8") as output:
    for item in quarantine:
        output.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"Lines read: {lines_read}")
print(f"Passed: {len(valid_bookings)}")
print(f"Quarantined: {len(quarantine)}")
print("Quarantined by reason:")
for reason, count in sorted(quarantine_counts.items()):
    print(f"  {reason}: {count}")

Lines read: 283
Passed: 259
Quarantined: 24
Quarantined by reason:
  booking: Value error, booked_at must be before the event date: 3
  booking: Value error, total must equal the sum of ticket qty times price: 4
  duplicate booking_id: 3
  event: Input should be a valid dictionary or instance of Event: 2
  student.id: String should match pattern '^\d{2}-\d{4}$': 4
  tickets.0.qty: Input should be greater than or equal to 1: 3
  tickets.0.type: Input should be 'standard', 'vip' or 'student': 3
  tickets: List should have at least 1 item after validation, not 0: 2


## Part B — Flatten nested records

The source-only `promo_code` field is modeled as optional. Its absence is represented as null in the flat bookings table.

In [4]:
base_fields = {
    "booking_id",
    "booked_at",
    "student",
    "event",
    "tickets",
    "total",
    "payment",
    "status",
}
all_fields = set().union(
    *(set(document) for _, document in raw_documents if isinstance(document, dict))
)
drift_fields = sorted(all_fields - base_fields)
drift_summary = {}
for field in drift_fields:
    first_line, first_document = next(
        (line_number, document)
        for line_number, document in raw_documents
        if isinstance(document, dict) and field in document
    )
    drift_summary[field] = {
        "first_line": first_line,
        "first_booking_id": first_document.get("booking_id"),
        "first_booked_at": first_document.get("booked_at"),
        "earliest_booked_at": min(
            document["booked_at"]
            for _, document in raw_documents
            if isinstance(document, dict)
            and field in document
            and "booked_at" in document
        ),
        "documents_with_field": sum(
            1
            for _, document in raw_documents
            if isinstance(document, dict) and field in document
        ),
    }
print("Schema drift:", drift_summary)

booking_rows = []
ticket_rows = []
for booking in valid_bookings:
    booking_rows.append(
        {
            "booking_id": booking.booking_id,
            "booked_at": booking.booked_at,
            "student_id": booking.student.id,
            "student_name": booking.student.name,
            "student_faculty": booking.student.faculty,
            "event_id": booking.event.id,
            "event_title": booking.event.title,
            "event_club": booking.event.club,
            "event_date": booking.event.date,
            "event_venue": booking.event.venue,
            "total": booking.total,
            "payment": booking.payment,
            "status": booking.status,
            "promo_code": booking.promo_code,
        }
    )
    for ticket in booking.tickets:
        ticket_rows.append(
            {
                "booking_id": booking.booking_id,
                "type": ticket.type,
                "qty": ticket.qty,
                "price": ticket.price,
                "amount": ticket.qty * ticket.price,
            }
        )

bookings = pd.DataFrame(booking_rows)
booking_tickets = pd.DataFrame(ticket_rows)
assert bookings["booking_id"].is_unique

ticket_totals = {}
for row in ticket_rows:
    ticket_totals[row["booking_id"]] = (
        ticket_totals.get(row["booking_id"], Decimal("0")) + row["amount"]
    )
assert all(ticket_totals[row["booking_id"]] == row["total"] for row in booking_rows)
print(f"Bookings: {len(bookings)} rows; booking_tickets: {len(booking_tickets)} rows")

Schema drift: {'promo_code': {'first_line': 4, 'first_booking_id': 'B0004', 'first_booked_at': '2026-11-01T06:42:00Z', 'earliest_booked_at': '2026-10-20T01:05:00Z', 'documents_with_field': 193}}
Bookings: 259 rows; booking_tickets: 314 rows


## Part C — DuckDB star schema and Parquet

The fact grain is one ticket entry. Booking totals remain in the bookings table and are not repeated on each fact row.

In [5]:
connection = duckdb.connect(database=":memory:")
connection.register("bookings_flat", bookings)
connection.register("booking_tickets_flat", booking_tickets)

connection.execute("""
    CREATE TABLE dim_event AS
    SELECT ROW_NUMBER() OVER (ORDER BY event_id)::BIGINT AS event_key,
           event_id, event_title, event_club, event_date, event_venue
    FROM (
        SELECT event_id, event_title, event_club, event_date, event_venue
        FROM bookings_flat
        QUALIFY ROW_NUMBER() OVER (
            PARTITION BY event_id ORDER BY event_title, event_club, event_date, event_venue
        ) = 1
    )
""")
connection.execute("""
    CREATE TABLE dim_student AS
    SELECT ROW_NUMBER() OVER (ORDER BY student_id)::BIGINT AS student_key,
           student_id, student_name, student_faculty
    FROM (
        SELECT student_id, student_name, student_faculty
        FROM bookings_flat
        QUALIFY ROW_NUMBER() OVER (
            PARTITION BY student_id ORDER BY student_name, student_faculty
        ) = 1
    )
""")
connection.execute("""
    CREATE TABLE fact_ticket_sales AS
    SELECT bt.booking_id, de.event_key, ds.student_key,
           bt.type AS ticket_type, bt.qty, bt.price, bt.amount,
           b.status,
           CAST(strftime(CAST(b.booked_at AS TIMESTAMP), '%Y%m%d') AS INTEGER) AS booked_date_key
    FROM booking_tickets_flat AS bt
    JOIN bookings_flat AS b USING (booking_id)
    JOIN dim_event AS de ON b.event_id = de.event_id
    JOIN dim_student AS ds ON b.student_id = ds.student_id
""")
assert connection.execute(
    "SELECT COUNT(*) = COUNT(DISTINCT event_id) FROM dim_event"
).fetchone()[0]
assert connection.execute(
    "SELECT COUNT(*) = COUNT(DISTINCT student_id) FROM dim_student"
).fetchone()[0]

fact_rows, fact_amount = connection.execute(
    "SELECT COUNT(*), SUM(amount) FROM fact_ticket_sales"
).fetchone()
bookings_total = sum((row["total"] for row in booking_rows), Decimal("0"))
assert fact_rows == len(booking_tickets)
assert Decimal(str(fact_amount)) == bookings_total

confirmed_sales = connection.execute("""
    SELECT de.event_club AS club, f.ticket_type,
           SUM(f.qty) AS tickets_sold, SUM(f.amount) AS revenue
    FROM fact_ticket_sales AS f
    JOIN dim_event AS de USING (event_key)
    WHERE f.status = 'confirmed'
    GROUP BY de.event_club, f.ticket_type
    ORDER BY de.event_club, f.ticket_type
""").fetchdf()
display(confirmed_sales)

star_dir = Path("star")
if star_dir.exists():
    import shutil

    shutil.rmtree(star_dir)
star_dir.mkdir()


def sql_path(path: Path) -> str:
    return str(path.resolve()).replace("\\", "/").replace("'", "''")


connection.execute(
    f"COPY dim_event TO '{sql_path(star_dir / 'dim_event.parquet')}' (FORMAT PARQUET)"
)
connection.execute(
    f"COPY dim_student TO '{sql_path(star_dir / 'dim_student.parquet')}' (FORMAT PARQUET)"
)
connection.execute(f"""
    COPY (
        SELECT f.*, strftime(CAST(b.booked_at AS TIMESTAMP), '%Y-%m') AS booked_month
        FROM fact_ticket_sales AS f
        JOIN bookings_flat AS b USING (booking_id)
    ) TO '{sql_path(star_dir / 'fact_ticket_sales')}'
    (FORMAT PARQUET, PARTITION_BY (booked_month))
""")
print(f"Saved dimensions and month-partitioned fact files under {star_dir.resolve()}")

,club,ticket_type,tickets_sold,revenue
0,Chess,standard,45.0,900.0
1,Chess,student,44.0,440.0
2,Debate,standard,65.0,1950.0
3,Debate,student,34.0,510.0
4,Debate,vip,40.0,3000.0
5,Football,standard,64.0,1600.0
6,Music,standard,67.0,2930.0
7,Music,student,20.0,1000.0
8,Music,vip,78.0,9990.0
9,Robotics,standard,60.0,2400.0


Saved dimensions and month-partitioned fact files under C:\Users\bongo\Desktop\yousef-elrayes_58-0924_IET-02_task2\star
